# Macroeconomic Indicators and Stock Market Returns
## Forecasting and Causal Relationships: An Econometric Analysis in R

---

### Project Overview

This notebook presents a comprehensive econometric analysis examining the relationships between macroeconomic indicators and U.S. stock market returns. We investigate:

1. **Temporal dynamics** between S&P 500 returns and key macroeconomic variables
2. **Granger causality** to determine predictive relationships
3. **Vector Autoregression (VAR)** models for multivariate time series analysis
4. **Impulse Response Functions (IRFs)** to understand shock propagation
5. **Forecasting performance** using out-of-sample evaluation

### Data Sources
- **S&P 500 Index**: Yahoo Finance (via `quantmod`)
- **Macroeconomic Indicators**: Federal Reserve Economic Data (FRED) via `fredr` package
  - Consumer Price Index (CPI)
  - Federal Funds Rate
  - Unemployment Rate
  - Industrial Production Index

### Methodology
Data Ingest → Cleaning → Exploratory Analysis → Stationarity Tests → Model Selection → Estimation → Diagnostics → Evaluation → Interpretation

---

**Author**: Econometric Analysis Project  
**Runtime**: Google Colab with R via rpy2  
**Last Updated**: 2025

---
## Part 1: Environment Setup
### 1.1 Install R and rpy2 in Google Colab

First, we need to install R and the rpy2 interface to run R code within Python cells.

In [ ]:
# Install R and rpy2 for running R in Colab
import subprocess
import sys

# Update and install R
print("Installing R...")
subprocess.run(['apt-get', 'update', '-qq'], check=True, capture_output=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'r-base', 'r-base-dev'], check=True, capture_output=True)
print("R installed successfully!")

# Install rpy2
print("Installing rpy2...")
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'rpy2'], check=True, capture_output=True)
print("rpy2 installed successfully!")

# Verify installations
result = subprocess.run(['R', '--version'], capture_output=True, text=True)
print("\nR Version:")
print(result.stdout.split('\n')[0])

In [ ]:
# Load rpy2 and enable R magic commands
%load_ext rpy2.ipython

print("R magic commands enabled! Use %%R at the start of a cell to run R code.")

### 1.2 Install Required R Packages

We install all necessary R packages for econometric analysis, data retrieval, and visualization.

In [ ]:
%%R
# Set CRAN mirror
options(repos = c(CRAN = "https://cloud.r-project.org/"))

# Function to install packages if not already installed
install_if_missing <- function(packages) {
  new_packages <- packages[!(packages %in% installed.packages()[, "Package"])]
  if (length(new_packages) > 0) {
    install.packages(new_packages, dependencies = TRUE, quiet = TRUE)
  }
}

# List of required packages
required_packages <- c(
  # Data retrieval
  "quantmod",      # Yahoo Finance data
  "fredr",         # FRED data (we'll use alternative method)
  "pdfetch",       # Alternative FRED access without API key
  
  # Data manipulation
  "dplyr",
  "tidyr",
  "zoo",
  "xts",
  "lubridate",
  
  # Econometrics
  "tseries",       # Time series tests
  "urca",          # Unit root and cointegration tests
  "vars",          # VAR models
  "lmtest",        # Diagnostic tests
  "sandwich",      # Robust standard errors
  "forecast",      # Forecasting
  "dynlm",         # Dynamic linear models
  "strucchange",   # Structural change tests
  
  # Visualization
  "ggplot2",
  "gridExtra",
  "corrplot",
  "scales"
)

cat("Installing required packages...\n")
install_if_missing(required_packages)
cat("Package installation complete!\n")

In [ ]:
%%R
# Load all packages
suppressPackageStartupMessages({
  library(quantmod)
  library(dplyr)
  library(tidyr)
  library(zoo)
  library(xts)
  library(lubridate)
  library(tseries)
  library(urca)
  library(vars)
  library(lmtest)
  library(sandwich)
  library(forecast)
  library(ggplot2)
  library(gridExtra)
  library(corrplot)
  library(scales)
})

# Set seed for reproducibility
set.seed(42)

cat("All packages loaded successfully!\n")
cat("\nSession Information:\n")
cat("===================\n")
sessionInfo()

---
## Part 2: Data Acquisition

### 2.1 Fetch S&P 500 Data from Yahoo Finance

We retrieve S&P 500 index data using the `quantmod` package, which provides a convenient interface to Yahoo Finance.

In [ ]:
%%R
# Define date range for analysis
start_date <- "2000-01-01"
end_date <- Sys.Date()

cat("Fetching S&P 500 data from Yahoo Finance...\n")
cat(paste("Date range:", start_date, "to", end_date, "\n\n"))

# Fetch S&P 500 data (^GSPC is the Yahoo Finance symbol)
tryCatch({
  getSymbols("^GSPC", src = "yahoo", from = start_date, to = end_date, auto.assign = TRUE)
  sp500_raw <- GSPC
  cat("S&P 500 data retrieved successfully!\n")
  cat(paste("Observations:", nrow(sp500_raw), "\n"))
  cat(paste("Date range:", min(index(sp500_raw)), "to", max(index(sp500_raw)), "\n\n"))
  
  # Display first few rows
  cat("First observations:\n")
  head(sp500_raw)
}, error = function(e) {
  cat(paste("Error fetching data:", e$message, "\n"))
})

### 2.2 Fetch Macroeconomic Data from FRED

We retrieve macroeconomic indicators directly from FRED using the `quantmod` package's `getSymbols` function with FRED as the source. This method doesn't require an API key for basic series.

In [ ]:
%%R
# FRED series codes
fred_series <- list(
  CPIAUCSL = "Consumer Price Index (All Urban Consumers)",
  FEDFUNDS = "Federal Funds Effective Rate",
  UNRATE = "Unemployment Rate",
  INDPRO = "Industrial Production Index"
)

cat("Fetching macroeconomic data from FRED...\n\n")

# Fetch each series
macro_data <- list()

for (series_id in names(fred_series)) {
  cat(paste("Fetching", series_id, "-", fred_series[[series_id]], "...\n"))
  tryCatch({
    getSymbols(series_id, src = "FRED", auto.assign = TRUE)
    macro_data[[series_id]] <- get(series_id)
    cat(paste("  Success! Observations:", nrow(macro_data[[series_id]]), "\n"))
  }, error = function(e) {
    cat(paste("  Error:", e$message, "\n"))
  })
}

cat("\nMacroeconomic data retrieval complete!\n")

In [ ]:
%%R
# Display summary of retrieved macro data
cat("Summary of Macroeconomic Variables:\n")
cat("===================================\n\n")

for (series_id in names(macro_data)) {
  data <- macro_data[[series_id]]
  cat(paste0(series_id, " (", fred_series[[series_id]], "):\n"))
  cat(paste("  Period:", min(index(data)), "to", max(index(data)), "\n"))
  cat(paste("  Observations:", nrow(data), "\n"))
  cat(paste("  Mean:", round(mean(data, na.rm = TRUE), 4), "\n"))
  cat(paste("  Std Dev:", round(sd(data, na.rm = TRUE), 4), "\n\n"))
}

---
## Part 3: Data Cleaning and Preparation

### 3.1 Calculate Stock Returns

We calculate log returns for the S&P 500 index, which are preferred in financial econometrics due to their time-additivity and approximate normality properties.

In [ ]:
%%R
# Extract adjusted closing prices and calculate returns
sp500_prices <- Ad(sp500_raw)  # Adjusted close prices
colnames(sp500_prices) <- "SP500"

# Calculate log returns (daily)
sp500_returns_daily <- diff(log(sp500_prices)) * 100  # Percentage returns
sp500_returns_daily <- sp500_returns_daily[-1]  # Remove first NA
colnames(sp500_returns_daily) <- "SP500_Return"

cat("S&P 500 Daily Returns Summary:\n")
cat("==============================\n")
summary(sp500_returns_daily)

cat("\nAnnualized Statistics:\n")
cat(paste("  Mean Annual Return:", round(mean(sp500_returns_daily, na.rm = TRUE) * 252, 2), "%\n"))
cat(paste("  Annual Volatility:", round(sd(sp500_returns_daily, na.rm = TRUE) * sqrt(252), 2), "%\n"))
cat(paste("  Skewness:", round(skewness(as.numeric(sp500_returns_daily)), 4), "\n"))
cat(paste("  Kurtosis:", round(kurtosis(as.numeric(sp500_returns_daily)), 4), "\n"))

In [ ]:
%%R
# Convert daily returns to monthly for alignment with macro data
# Most macro data is monthly

# Aggregate to monthly returns
sp500_monthly <- to.monthly(sp500_prices, indexAt = "lastof", OHLC = FALSE)
sp500_returns_monthly <- diff(log(sp500_monthly)) * 100
sp500_returns_monthly <- sp500_returns_monthly[-1]
colnames(sp500_returns_monthly) <- "SP500_Return"

# Convert index to Date class for merging
index(sp500_returns_monthly) <- as.Date(index(sp500_returns_monthly))

cat("S&P 500 Monthly Returns Summary:\n")
cat("================================\n")
summary(sp500_returns_monthly)

cat(paste("\nNumber of monthly observations:", nrow(sp500_returns_monthly), "\n"))

### 3.2 Process Macroeconomic Variables

We transform the macroeconomic variables to ensure stationarity:
- **CPI**: Convert to inflation rate (percentage change)
- **Fed Funds Rate**: First difference
- **Unemployment Rate**: First difference
- **Industrial Production**: Growth rate (percentage change)

In [ ]:
%%R
# Process CPI - Calculate year-over-year inflation rate
cpi <- macro_data$CPIAUCSL
inflation <- (cpi / lag(cpi, 12) - 1) * 100  # YoY inflation
colnames(inflation) <- "Inflation"

# Process Fed Funds Rate - Use level and first difference
fedfunds <- macro_data$FEDFUNDS
colnames(fedfunds) <- "FedFunds"
fedfunds_diff <- diff(fedfunds)
colnames(fedfunds_diff) <- "FedFunds_Diff"

# Process Unemployment Rate - First difference
unrate <- macro_data$UNRATE
colnames(unrate) <- "Unemployment"
unrate_diff <- diff(unrate)
colnames(unrate_diff) <- "Unemployment_Diff"

# Process Industrial Production - Month-over-month growth rate
indpro <- macro_data$INDPRO
indpro_growth <- diff(log(indpro)) * 100  # MoM growth
colnames(indpro_growth) <- "IndProd_Growth"

cat("Macroeconomic transformations complete:\n")
cat("  - Inflation: Year-over-year CPI change (%)\n")
cat("  - FedFunds: Level and first difference\n")
cat("  - Unemployment: First difference\n")
cat("  - Industrial Production: Month-over-month growth (%)\n")

### 3.3 Merge and Align Time Series

We merge all series into a single dataset, ensuring proper alignment of dates and handling missing values.

In [ ]:
%%R
# Merge all series
# First, ensure all indices are in the same format
index(inflation) <- as.Date(index(inflation))
index(fedfunds) <- as.Date(index(fedfunds))
index(fedfunds_diff) <- as.Date(index(fedfunds_diff))
index(unrate) <- as.Date(index(unrate))
index(unrate_diff) <- as.Date(index(unrate_diff))
index(indpro_growth) <- as.Date(index(indpro_growth))

# Merge all series
combined_data <- merge(
  sp500_returns_monthly,
  inflation,
  fedfunds,
  unrate,
  indpro_growth,
  all = FALSE  # Inner join - keep only dates where all series have data
)

# Remove rows with NA values
combined_data <- na.omit(combined_data)

cat("Combined Dataset Summary:\n")
cat("=========================\n")
cat(paste("Period:", min(index(combined_data)), "to", max(index(combined_data)), "\n"))
cat(paste("Observations:", nrow(combined_data), "\n"))
cat(paste("Variables:", ncol(combined_data), "\n\n"))

cat("Variable Names:\n")
print(colnames(combined_data))

cat("\nFirst observations:\n")
head(combined_data)

In [ ]:
%%R
# Store intermediate dataset
econ_data <- combined_data

# Create a data frame version for certain analyses
econ_df <- data.frame(
  Date = index(econ_data),
  coredata(econ_data)
)

cat("Descriptive Statistics:\n")
cat("=======================\n\n")

# Calculate comprehensive statistics
stats_summary <- data.frame(
  Variable = colnames(econ_data),
  Mean = apply(econ_data, 2, mean, na.rm = TRUE),
  Median = apply(econ_data, 2, median, na.rm = TRUE),
  Std_Dev = apply(econ_data, 2, sd, na.rm = TRUE),
  Min = apply(econ_data, 2, min, na.rm = TRUE),
  Max = apply(econ_data, 2, max, na.rm = TRUE),
  Skewness = apply(econ_data, 2, function(x) skewness(as.numeric(x))),
  Kurtosis = apply(econ_data, 2, function(x) kurtosis(as.numeric(x)))
)
rownames(stats_summary) <- NULL

print(round(stats_summary, 4))

---
## Part 4: Exploratory Data Analysis

### 4.1 Time Series Visualization

In [ ]:
%%R
# Set up plotting parameters
options(repr.plot.width = 14, repr.plot.height = 10)

# Create time series plots
par(mfrow = c(3, 2), mar = c(4, 4, 3, 1))

# S&P 500 Returns
plot(econ_data$SP500_Return, main = "S&P 500 Monthly Returns",
     ylab = "Return (%)", xlab = "", col = "darkblue", lwd = 1)
abline(h = 0, col = "red", lty = 2)
abline(h = mean(econ_data$SP500_Return), col = "green", lty = 2)

# Inflation
plot(econ_data$Inflation, main = "Inflation Rate (YoY)",
     ylab = "Inflation (%)", xlab = "", col = "darkred", lwd = 1)
abline(h = 2, col = "green", lty = 2)  # Fed target

# Federal Funds Rate
plot(econ_data$FedFunds, main = "Federal Funds Rate",
     ylab = "Rate (%)", xlab = "", col = "darkgreen", lwd = 1)

# Unemployment Rate
plot(econ_data$Unemployment, main = "Unemployment Rate",
     ylab = "Rate (%)", xlab = "", col = "darkorange", lwd = 1)

# Industrial Production Growth
plot(econ_data$IndProd_Growth, main = "Industrial Production Growth (MoM)",
     ylab = "Growth (%)", xlab = "", col = "purple", lwd = 1)
abline(h = 0, col = "red", lty = 2)

# Empty plot for legend
plot.new()
legend("center", 
       legend = c("S&P 500 Returns", "Inflation", "Fed Funds Rate", 
                  "Unemployment", "Industrial Production"),
       col = c("darkblue", "darkred", "darkgreen", "darkorange", "purple"),
       lwd = 2, cex = 1.2, bty = "n")

### 4.2 Correlation Analysis

In [ ]:
%%R
# Calculate correlation matrix
cor_matrix <- cor(econ_data, use = "pairwise.complete.obs")

cat("Correlation Matrix:\n")
cat("===================\n\n")
print(round(cor_matrix, 4))

# Visualize correlation matrix
par(mfrow = c(1, 1), mar = c(1, 1, 2, 1))
corrplot(cor_matrix, method = "color", type = "upper",
         addCoef.col = "black", number.cex = 0.8,
         tl.col = "black", tl.srt = 45,
         title = "Correlation Matrix of Economic Variables",
         mar = c(0, 0, 2, 0))

In [ ]:
%%R
# Test significance of correlations
cat("Statistical Significance of Correlations with S&P 500 Returns:\n")
cat("================================================================\n\n")

for (var in colnames(econ_data)[-1]) {
  test <- cor.test(as.numeric(econ_data$SP500_Return), 
                   as.numeric(econ_data[, var]))
  cat(paste0(var, ":\n"))
  cat(paste("  Correlation:", round(test$estimate, 4), "\n"))
  cat(paste("  t-statistic:", round(test$statistic, 4), "\n"))
  cat(paste("  p-value:", format.pval(test$p.value, digits = 4), "\n"))
  cat(paste("  Significant:", ifelse(test$p.value < 0.05, "Yes***", "No"), "\n\n"))
}

### 4.3 Autocorrelation Analysis

In [ ]:
%%R
# ACF and PACF plots for S&P 500 returns
par(mfrow = c(2, 2), mar = c(4, 4, 3, 1))

# S&P 500 Returns
acf(as.numeric(econ_data$SP500_Return), main = "ACF: S&P 500 Returns", lag.max = 24)
pacf(as.numeric(econ_data$SP500_Return), main = "PACF: S&P 500 Returns", lag.max = 24)

# Squared returns (for volatility clustering)
acf(as.numeric(econ_data$SP500_Return)^2, main = "ACF: Squared Returns", lag.max = 24)
pacf(as.numeric(econ_data$SP500_Return)^2, main = "PACF: Squared Returns", lag.max = 24)

In [ ]:
%%R
# Ljung-Box test for autocorrelation
cat("Ljung-Box Test for Autocorrelation:\n")
cat("====================================\n\n")

for (var in colnames(econ_data)) {
  lb_test <- Box.test(as.numeric(econ_data[, var]), lag = 12, type = "Ljung-Box")
  cat(paste0(var, ":\n"))
  cat(paste("  Chi-squared:", round(lb_test$statistic, 4), "\n"))
  cat(paste("  p-value:", format.pval(lb_test$p.value, digits = 4), "\n"))
  cat(paste("  Autocorrelated:", ifelse(lb_test$p.value < 0.05, "Yes", "No"), "\n\n"))
}

---
## Part 5: Stationarity Testing

Stationarity is a crucial assumption for many time series models. We apply three different unit root tests:
1. **Augmented Dickey-Fuller (ADF)** test
2. **Phillips-Perron (PP)** test
3. **KPSS** test (null hypothesis is stationarity)

### 5.1 Augmented Dickey-Fuller Test

In [ ]:
%%R
cat("Augmented Dickey-Fuller (ADF) Test Results:\n")
cat("============================================\n")
cat("H0: Series has a unit root (non-stationary)\n")
cat("H1: Series is stationary\n\n")

adf_results <- data.frame(
  Variable = character(),
  Test_Statistic = numeric(),
  P_Value = numeric(),
  Lags = integer(),
  Conclusion = character(),
  stringsAsFactors = FALSE
)

for (var in colnames(econ_data)) {
  adf_test <- adf.test(as.numeric(econ_data[, var]), alternative = "stationary")
  
  conclusion <- ifelse(adf_test$p.value < 0.05, "Stationary", "Non-stationary")
  
  adf_results <- rbind(adf_results, data.frame(
    Variable = var,
    Test_Statistic = round(adf_test$statistic, 4),
    P_Value = round(adf_test$p.value, 4),
    Lags = adf_test$parameter,
    Conclusion = conclusion
  ))
}

print(adf_results)

### 5.2 Phillips-Perron Test

In [ ]:
%%R
cat("Phillips-Perron (PP) Test Results:\n")
cat("===================================\n")
cat("H0: Series has a unit root (non-stationary)\n")
cat("H1: Series is stationary\n\n")

pp_results <- data.frame(
  Variable = character(),
  Test_Statistic = numeric(),
  P_Value = numeric(),
  Conclusion = character(),
  stringsAsFactors = FALSE
)

for (var in colnames(econ_data)) {
  pp_test <- pp.test(as.numeric(econ_data[, var]), alternative = "stationary")
  
  conclusion <- ifelse(pp_test$p.value < 0.05, "Stationary", "Non-stationary")
  
  pp_results <- rbind(pp_results, data.frame(
    Variable = var,
    Test_Statistic = round(pp_test$statistic, 4),
    P_Value = round(pp_test$p.value, 4),
    Conclusion = conclusion
  ))
}

print(pp_results)

### 5.3 KPSS Test

In [ ]:
%%R
cat("KPSS Test Results:\n")
cat("==================\n")
cat("H0: Series is stationary\n")
cat("H1: Series has a unit root (non-stationary)\n\n")

kpss_results <- data.frame(
  Variable = character(),
  Test_Statistic = numeric(),
  P_Value = character(),
  Conclusion = character(),
  stringsAsFactors = FALSE
)

for (var in colnames(econ_data)) {
  kpss_test <- kpss.test(as.numeric(econ_data[, var]), null = "Level")
  
  # KPSS has different p-value interpretation
  conclusion <- ifelse(kpss_test$p.value > 0.05, "Stationary", "Non-stationary")
  
  kpss_results <- rbind(kpss_results, data.frame(
    Variable = var,
    Test_Statistic = round(kpss_test$statistic, 4),
    P_Value = ifelse(kpss_test$p.value < 0.01, "< 0.01",
                     ifelse(kpss_test$p.value > 0.1, "> 0.10", 
                            as.character(round(kpss_test$p.value, 4)))),
    Conclusion = conclusion
  ))
}

print(kpss_results)

### 5.4 Summary of Stationarity Tests

In [ ]:
%%R
cat("\n========================================\n")
cat("SUMMARY OF STATIONARITY TEST RESULTS\n")
cat("========================================\n\n")

summary_results <- data.frame(
  Variable = adf_results$Variable,
  ADF = adf_results$Conclusion,
  PP = pp_results$Conclusion,
  KPSS = kpss_results$Conclusion
)

print(summary_results)

cat("\nInterpretation:\n")
cat("--------------\n")
cat("- Variables showing 'Stationary' in majority of tests can be used in levels.\n")
cat("- Variables showing 'Non-stationary' may need differencing.\n")
cat("- FedFunds and Unemployment typically need differencing for VAR analysis.\n")

### 5.5 Transform Non-Stationary Variables

In [ ]:
%%R
# Create dataset with stationary transformations
# Based on test results, we'll difference FedFunds and Unemployment

econ_stationary <- merge(
  econ_data$SP500_Return,
  econ_data$Inflation,
  diff(econ_data$FedFunds),
  diff(econ_data$Unemployment),
  econ_data$IndProd_Growth,
  all = FALSE
)

colnames(econ_stationary) <- c("SP500_Return", "Inflation", "FedFunds_Diff", 
                                "Unemployment_Diff", "IndProd_Growth")

econ_stationary <- na.omit(econ_stationary)

cat("Stationary Dataset Created:\n")
cat("===========================\n")
cat(paste("Observations:", nrow(econ_stationary), "\n"))
cat(paste("Variables:", ncol(econ_stationary), "\n\n"))

cat("Verify stationarity of transformed variables:\n")
for (var in c("FedFunds_Diff", "Unemployment_Diff")) {
  adf_test <- adf.test(as.numeric(econ_stationary[, var]))
  cat(paste0(var, ": ADF p-value = ", round(adf_test$p.value, 4), 
             " (", ifelse(adf_test$p.value < 0.05, "Stationary", "Non-stationary"), ")\n"))
}

---
## Part 6: VAR Model Selection

### 6.1 Optimal Lag Selection

We use information criteria (AIC, BIC, HQ) to determine the optimal lag length for our VAR model.

In [ ]:
%%R
# Convert to matrix for VAR estimation
var_data <- as.matrix(econ_stationary)

# Select optimal lag length
cat("VAR Lag Order Selection:\n")
cat("========================\n\n")

lag_selection <- VARselect(var_data, lag.max = 12, type = "const")

cat("Information Criteria Values:\n")
print(lag_selection$criteria)

cat("\nOptimal Lag by Criterion:\n")
print(lag_selection$selection)

# Choose lag based on AIC (often preferred for forecasting)
optimal_lag <- lag_selection$selection["AIC(n)"]
cat(paste("\nSelected lag order (AIC):", optimal_lag, "\n"))

In [ ]:
%%R
# Visualize information criteria
par(mfrow = c(1, 1), mar = c(4, 4, 3, 1))

criteria_df <- data.frame(
  Lag = 1:ncol(lag_selection$criteria),
  AIC = lag_selection$criteria["AIC(n)", ],
  BIC = lag_selection$criteria["SC(n)", ],
  HQ = lag_selection$criteria["HQ(n)", ]
)

# Normalize for comparison
criteria_norm <- criteria_df
criteria_norm$AIC <- scale(criteria_df$AIC)
criteria_norm$BIC <- scale(criteria_df$BIC)
criteria_norm$HQ <- scale(criteria_df$HQ)

plot(criteria_norm$Lag, criteria_norm$AIC, type = "b", col = "blue", 
     ylim = range(c(criteria_norm$AIC, criteria_norm$BIC, criteria_norm$HQ)),
     xlab = "Lag Order", ylab = "Standardized Information Criterion",
     main = "VAR Lag Order Selection Criteria", pch = 19)
lines(criteria_norm$Lag, criteria_norm$BIC, type = "b", col = "red", pch = 17)
lines(criteria_norm$Lag, criteria_norm$HQ, type = "b", col = "green", pch = 15)
legend("topright", legend = c("AIC", "BIC", "HQ"), 
       col = c("blue", "red", "green"), pch = c(19, 17, 15), lty = 1)

---
## Part 7: VAR Model Estimation

### 7.1 Estimate VAR Model

In [ ]:
%%R
# Estimate VAR model with optimal lag
# Use a reasonable lag (min of AIC suggestion and 6 to avoid overfitting)
p <- min(optimal_lag, 6)

cat(paste("Estimating VAR(", p, ") model...\n\n", sep = ""))

var_model <- VAR(var_data, p = p, type = "const")

cat("VAR Model Summary:\n")
cat("==================\n")
summary(var_model)

In [ ]:
%%R
# Extract coefficients for S&P 500 equation
cat("\nCoefficients for S&P 500 Returns Equation:\n")
cat("==========================================\n")

sp500_eq <- var_model$varresult$SP500_Return
coef_summary <- summary(sp500_eq)$coefficients

# Format output
coef_df <- data.frame(
  Variable = rownames(coef_summary),
  Estimate = round(coef_summary[, 1], 4),
  Std_Error = round(coef_summary[, 2], 4),
  t_value = round(coef_summary[, 3], 4),
  p_value = round(coef_summary[, 4], 4),
  Significance = ifelse(coef_summary[, 4] < 0.01, "***",
                        ifelse(coef_summary[, 4] < 0.05, "**",
                               ifelse(coef_summary[, 4] < 0.1, "*", "")))
)
rownames(coef_df) <- NULL
print(coef_df)

cat("\nSignificance codes: *** p<0.01, ** p<0.05, * p<0.1\n")

### 7.2 Model Fit Statistics

In [ ]:
%%R
cat("Model Fit Statistics by Equation:\n")
cat("==================================\n\n")

fit_stats <- data.frame(
  Equation = character(),
  R_squared = numeric(),
  Adj_R_squared = numeric(),
  F_statistic = numeric(),
  p_value = numeric(),
  stringsAsFactors = FALSE
)

for (eq_name in names(var_model$varresult)) {
  eq <- var_model$varresult[[eq_name]]
  eq_summary <- summary(eq)
  
  fit_stats <- rbind(fit_stats, data.frame(
    Equation = eq_name,
    R_squared = round(eq_summary$r.squared, 4),
    Adj_R_squared = round(eq_summary$adj.r.squared, 4),
    F_statistic = round(eq_summary$fstatistic[1], 4),
    p_value = round(pf(eq_summary$fstatistic[1], 
                       eq_summary$fstatistic[2], 
                       eq_summary$fstatistic[3], 
                       lower.tail = FALSE), 6)
  ))
}

print(fit_stats)

---
## Part 8: Granger Causality Tests

Granger causality tests whether lagged values of one variable help predict another variable, beyond the predictive power of the target's own lagged values.

In [ ]:
%%R
cat("Granger Causality Tests:\n")
cat("========================\n")
cat("H0: Variable does NOT Granger-cause the target\n")
cat("H1: Variable Granger-causes the target\n\n")

# Test causality to S&P 500 returns
cat("=== Testing causality TO S&P 500 Returns ===\n\n")

granger_to_sp500 <- data.frame(
  Cause = character(),
  F_statistic = numeric(),
  df1 = integer(),
  df2 = integer(),
  p_value = numeric(),
  Conclusion = character(),
  stringsAsFactors = FALSE
)

for (cause_var in colnames(econ_stationary)[-1]) {
  gc_test <- causality(var_model, cause = cause_var)$Granger
  
  conclusion <- ifelse(gc_test$p.value < 0.05, 
                       paste(cause_var, "Granger-causes SP500"),
                       "No Granger causality")
  
  granger_to_sp500 <- rbind(granger_to_sp500, data.frame(
    Cause = cause_var,
    F_statistic = round(gc_test$statistic, 4),
    df1 = gc_test$parameter[1],
    df2 = gc_test$parameter[2],
    p_value = round(gc_test$p.value, 4),
    Conclusion = conclusion
  ))
}

print(granger_to_sp500)

In [ ]:
%%R
# Test causality FROM S&P 500 returns to other variables
cat("\n=== Testing causality FROM S&P 500 Returns ===\n\n")

granger_from_sp500 <- causality(var_model, cause = "SP500_Return")$Granger

cat("Joint test that S&P 500 Returns Granger-causes all other variables:\n")
cat(paste("  F-statistic:", round(granger_from_sp500$statistic, 4), "\n"))
cat(paste("  df:", granger_from_sp500$parameter[1], ",", granger_from_sp500$parameter[2], "\n"))
cat(paste("  p-value:", round(granger_from_sp500$p.value, 4), "\n"))
cat(paste("  Conclusion:", ifelse(granger_from_sp500$p.value < 0.05,
                                   "SP500 Granger-causes other variables",
                                   "No significant Granger causality from SP500"), "\n"))

In [ ]:
%%R
# Comprehensive Granger causality matrix
cat("\n=== Complete Granger Causality Matrix ===\n")
cat("(p-values; * indicates significance at 5% level)\n\n")

vars_list <- colnames(econ_stationary)
n_vars <- length(vars_list)

gc_matrix <- matrix(NA, n_vars, n_vars)
rownames(gc_matrix) <- vars_list
colnames(gc_matrix) <- vars_list

for (i in 1:n_vars) {
  for (j in 1:n_vars) {
    if (i != j) {
      tryCatch({
        gc_test <- causality(var_model, cause = vars_list[i])$Granger
        gc_matrix[i, j] <- gc_test$p.value
      }, error = function(e) {
        gc_matrix[i, j] <- NA
      })
    }
  }
}

# Format matrix for display
gc_display <- matrix("", n_vars, n_vars)
rownames(gc_display) <- vars_list
colnames(gc_display) <- paste("→", vars_list)

for (i in 1:n_vars) {
  for (j in 1:n_vars) {
    if (i == j) {
      gc_display[i, j] <- "---"
    } else if (!is.na(gc_matrix[i, j])) {
      sig <- ifelse(gc_matrix[i, j] < 0.05, "*", "")
      gc_display[i, j] <- paste0(round(gc_matrix[i, j], 3), sig)
    }
  }
}

print(gc_display)

---
## Part 9: Impulse Response Functions

Impulse Response Functions (IRFs) trace the effect of a one-standard-deviation shock to one variable on current and future values of all variables in the system.

In [ ]:
%%R
# Compute Impulse Response Functions
cat("Computing Impulse Response Functions...\n")
cat("Horizon: 24 months\n")
cat("Confidence interval: 95%\n\n")

# IRF with bootstrapped confidence intervals
set.seed(42)  # For reproducibility
irf_results <- irf(var_model, n.ahead = 24, boot = TRUE, ci = 0.95, runs = 500)

cat("IRF computation complete!\n")

In [ ]:
%%R
# Plot IRFs - Response of S&P 500 to all shocks
par(mfrow = c(2, 3), mar = c(4, 4, 3, 1))

response_var <- "SP500_Return"
impulse_vars <- colnames(econ_stationary)

for (imp_var in impulse_vars) {
  # Extract IRF data
  irf_data <- irf_results$irf[[imp_var]][, response_var]
  lower <- irf_results$Lower[[imp_var]][, response_var]
  upper <- irf_results$Upper[[imp_var]][, response_var]
  
  # Plot
  plot(0:24, irf_data, type = "l", col = "blue", lwd = 2,
       ylim = range(c(lower, upper)),
       xlab = "Months", ylab = "Response",
       main = paste("Response of SP500 to", imp_var))
  
  # Add confidence bands
  polygon(c(0:24, 24:0), c(upper, rev(lower)), 
          col = rgb(0, 0, 1, 0.2), border = NA)
  
  # Add reference line
  abline(h = 0, col = "red", lty = 2)
}

# Add empty plot for spacing
plot.new()

In [ ]:
%%R
# Plot responses of other variables to S&P 500 shock
par(mfrow = c(2, 2), mar = c(4, 4, 3, 1))

impulse_var <- "SP500_Return"
response_vars <- colnames(econ_stationary)[-1]

for (resp_var in response_vars) {
  # Extract IRF data
  irf_data <- irf_results$irf[[impulse_var]][, resp_var]
  lower <- irf_results$Lower[[impulse_var]][, resp_var]
  upper <- irf_results$Upper[[impulse_var]][, resp_var]
  
  # Plot
  plot(0:24, irf_data, type = "l", col = "darkgreen", lwd = 2,
       ylim = range(c(lower, upper)),
       xlab = "Months", ylab = "Response",
       main = paste("Response of", resp_var, "to SP500"))
  
  # Add confidence bands
  polygon(c(0:24, 24:0), c(upper, rev(lower)), 
          col = rgb(0, 0.5, 0, 0.2), border = NA)
  
  # Add reference line
  abline(h = 0, col = "red", lty = 2)
}

---
## Part 10: Forecast Error Variance Decomposition

FEVD shows the proportion of the forecast error variance of each variable that can be attributed to shocks in each variable.

In [ ]:
%%R
# Compute Forecast Error Variance Decomposition
fevd_results <- fevd(var_model, n.ahead = 24)

cat("Forecast Error Variance Decomposition for S&P 500 Returns:\n")
cat("==========================================================\n\n")

# Extract FEVD for S&P 500
sp500_fevd <- fevd_results$SP500_Return

# Display at key horizons
horizons <- c(1, 3, 6, 12, 24)
fevd_display <- sp500_fevd[horizons, ] * 100  # Convert to percentages
rownames(fevd_display) <- paste0(horizons, " month(s)")

cat("Percentage of forecast error variance explained by:\n\n")
print(round(fevd_display, 2))

In [ ]:
%%R
# Visualize FEVD
par(mfrow = c(1, 1), mar = c(5, 4, 4, 8))

# Stacked bar plot
colors <- c("steelblue", "coral", "seagreen", "gold", "purple")

barplot(t(sp500_fevd * 100), 
        col = colors,
        border = "white",
        xlab = "Horizon (months)",
        ylab = "Variance Share (%)",
        main = "FEVD: S&P 500 Returns",
        names.arg = 1:24,
        las = 2)

legend("right", inset = c(-0.25, 0), xpd = TRUE,
       legend = colnames(sp500_fevd),
       fill = colors,
       title = "Shock Source",
       cex = 0.8)

In [ ]:
%%R
# FEVD summary table for all variables at 12-month horizon
cat("\nFEVD at 12-Month Horizon (All Variables):\n")
cat("==========================================\n\n")

fevd_12m <- matrix(NA, length(names(fevd_results)), ncol(sp500_fevd))
rownames(fevd_12m) <- names(fevd_results)
colnames(fevd_12m) <- colnames(sp500_fevd)

for (var in names(fevd_results)) {
  fevd_12m[var, ] <- fevd_results[[var]][12, ] * 100
}

print(round(fevd_12m, 2))

cat("\nNote: Rows show what % of each variable's forecast error is explained by column shocks.\n")

---
## Part 11: Model Diagnostics

### 11.1 Residual Analysis

In [ ]:
%%R
# Extract residuals
var_residuals <- residuals(var_model)
sp500_resid <- var_residuals[, "SP500_Return"]

# Residual diagnostics for S&P 500 equation
par(mfrow = c(2, 2), mar = c(4, 4, 3, 1))

# Time series plot of residuals
plot(sp500_resid, type = "l", col = "darkblue",
     main = "VAR Residuals: S&P 500 Returns",
     ylab = "Residual", xlab = "Time")
abline(h = 0, col = "red", lty = 2)

# Histogram
hist(sp500_resid, breaks = 30, freq = FALSE, col = "lightblue",
     main = "Distribution of Residuals",
     xlab = "Residual")
curve(dnorm(x, mean = mean(sp500_resid), sd = sd(sp500_resid)), 
      add = TRUE, col = "red", lwd = 2)

# Q-Q plot
qqnorm(sp500_resid, main = "Q-Q Plot", pch = 19, col = "darkblue")
qqline(sp500_resid, col = "red", lwd = 2)

# ACF of residuals
acf(sp500_resid, main = "ACF of Residuals", lag.max = 20)

### 11.2 Serial Correlation Tests

In [ ]:
%%R
cat("Serial Correlation Tests (VAR Residuals):\n")
cat("==========================================\n\n")

# Portmanteau test (multivariate Ljung-Box)
pt_test <- serial.test(var_model, lags.pt = 12, type = "PT.asymptotic")
cat("Portmanteau Test (asymptotic):\n")
cat(paste("  Chi-squared:", round(pt_test$serial$statistic, 4), "\n"))
cat(paste("  df:", pt_test$serial$parameter, "\n"))
cat(paste("  p-value:", round(pt_test$serial$p.value, 4), "\n"))
cat(paste("  Conclusion:", ifelse(pt_test$serial$p.value > 0.05, 
                                   "No serial correlation (good)",
                                   "Serial correlation detected"), "\n\n"))

# Breusch-Godfrey LM test
bg_test <- serial.test(var_model, lags.bg = 12, type = "BG")
cat("Breusch-Godfrey LM Test:\n")
cat(paste("  Chi-squared:", round(bg_test$serial$statistic, 4), "\n"))
cat(paste("  df:", bg_test$serial$parameter, "\n"))
cat(paste("  p-value:", round(bg_test$serial$p.value, 4), "\n"))
cat(paste("  Conclusion:", ifelse(bg_test$serial$p.value > 0.05,
                                   "No serial correlation (good)",
                                   "Serial correlation detected"), "\n"))

### 11.3 Heteroskedasticity Tests

In [ ]:
%%R
cat("Heteroskedasticity Tests:\n")
cat("=========================\n\n")

# ARCH-LM test (multivariate)
arch_test <- arch.test(var_model, lags.multi = 12)
cat("ARCH-LM Test (multivariate):\n")
cat(paste("  Chi-squared:", round(arch_test$arch.mul$statistic, 4), "\n"))
cat(paste("  df:", arch_test$arch.mul$parameter, "\n"))
cat(paste("  p-value:", round(arch_test$arch.mul$p.value, 4), "\n"))
cat(paste("  Conclusion:", ifelse(arch_test$arch.mul$p.value > 0.05,
                                   "No ARCH effects (good)",
                                   "ARCH effects detected"), "\n"))

### 11.4 Normality Tests

In [ ]:
%%R
cat("Normality Tests:\n")
cat("================\n\n")

# Jarque-Bera test (multivariate)
jb_test <- normality.test(var_model, multivariate.only = FALSE)

cat("Jarque-Bera Test (multivariate):\n")
cat(paste("  Chi-squared:", round(jb_test$jb.mul$JB$statistic, 4), "\n"))
cat(paste("  df:", jb_test$jb.mul$JB$parameter, "\n"))
cat(paste("  p-value:", round(jb_test$jb.mul$JB$p.value, 4), "\n"))
cat(paste("  Conclusion:", ifelse(jb_test$jb.mul$JB$p.value > 0.05,
                                   "Residuals are normally distributed",
                                   "Non-normality detected (common in financial data)"), "\n\n"))

# Individual equation normality
cat("Jarque-Bera Test (by equation):\n")
for (var in colnames(var_residuals)) {
  jb_uni <- jarque.bera.test(var_residuals[, var])
  cat(paste0("  ", var, ": p-value = ", round(jb_uni$p.value, 4),
             " (", ifelse(jb_uni$p.value > 0.05, "Normal", "Non-normal"), ")\n"))
}

### 11.5 Stability Analysis

In [ ]:
%%R
cat("VAR Model Stability:\n")
cat("====================\n\n")

# Check eigenvalues of companion matrix
roots <- roots(var_model)
cat("Eigenvalues of Companion Matrix:\n")
print(round(roots, 4))

cat(paste("\nMaximum eigenvalue modulus:", round(max(roots), 4), "\n"))
cat(paste("Stability condition (all < 1):", 
          ifelse(all(roots < 1), "SATISFIED - Model is stable", 
                 "VIOLATED - Model may be unstable"), "\n"))

In [ ]:
%%R
# Plot stability - eigenvalues in unit circle
par(mfrow = c(1, 1), mar = c(4, 4, 3, 1))

# Create unit circle
theta <- seq(0, 2*pi, length.out = 100)
plot(cos(theta), sin(theta), type = "l", col = "gray",
     xlim = c(-1.2, 1.2), ylim = c(-1.2, 1.2),
     xlab = "Real", ylab = "Imaginary",
     main = "VAR Stability: Eigenvalues and Unit Circle",
     asp = 1)
abline(h = 0, v = 0, col = "lightgray", lty = 2)

# Plot eigenvalues (as points on real axis since they're real)
points(roots, rep(0, length(roots)), pch = 19, col = "red", cex = 1.5)

legend("topright", legend = c("Unit Circle", "Eigenvalues"),
       col = c("gray", "red"), lty = c(1, NA), pch = c(NA, 19))

---
## Part 12: Forecasting

### 12.1 In-Sample Fit

In [ ]:
%%R
# Generate in-sample fitted values
fitted_values <- fitted(var_model)
actual_values <- var_data[(p+1):nrow(var_data), ]

# Plot actual vs fitted for S&P 500
par(mfrow = c(1, 1), mar = c(4, 4, 3, 1))

plot_dates <- index(econ_stationary)[(p+1):length(index(econ_stationary))]

plot(plot_dates, actual_values[, "SP500_Return"], type = "l", col = "darkblue",
     main = "S&P 500 Returns: Actual vs VAR Fitted",
     ylab = "Return (%)", xlab = "")
lines(plot_dates, fitted_values[, "SP500_Return"], col = "red", lty = 2)
legend("topright", legend = c("Actual", "Fitted"),
       col = c("darkblue", "red"), lty = c(1, 2))

### 12.2 Out-of-Sample Forecasting

In [ ]:
%%R
# Generate forecasts
forecast_horizon <- 12  # 12 months ahead

cat(paste("Generating", forecast_horizon, "month ahead forecasts...\n\n"))

var_forecast <- predict(var_model, n.ahead = forecast_horizon, ci = 0.95)

# Extract S&P 500 forecasts
sp500_fc <- var_forecast$fcst$SP500_Return

cat("S&P 500 Returns Forecast:\n")
cat("=========================\n\n")

fc_table <- data.frame(
  Horizon = 1:forecast_horizon,
  Forecast = round(sp500_fc[, "fcst"], 4),
  Lower_95 = round(sp500_fc[, "lower"], 4),
  Upper_95 = round(sp500_fc[, "upper"], 4)
)

print(fc_table)

In [ ]:
%%R
# Plot forecasts with fan chart
par(mfrow = c(1, 1), mar = c(4, 4, 3, 1))

# Historical data (last 60 months)
n_hist <- min(60, nrow(econ_stationary))
hist_data <- tail(as.numeric(econ_stationary$SP500_Return), n_hist)
hist_dates <- tail(index(econ_stationary), n_hist)

# Forecast dates
last_date <- max(index(econ_stationary))
fc_dates <- seq(last_date, by = "month", length.out = forecast_horizon + 1)[-1]

# Combine for plotting
all_dates <- c(hist_dates, fc_dates)
y_range <- range(c(hist_data, sp500_fc[, "lower"], sp500_fc[, "upper"]))

# Plot
plot(hist_dates, hist_data, type = "l", col = "darkblue", lwd = 1.5,
     xlim = range(all_dates), ylim = y_range,
     main = "S&P 500 Monthly Returns: History and Forecast",
     ylab = "Return (%)", xlab = "")

# Add forecast
lines(fc_dates, sp500_fc[, "fcst"], col = "red", lwd = 2)

# Add confidence interval
polygon(c(fc_dates, rev(fc_dates)),
        c(sp500_fc[, "upper"], rev(sp500_fc[, "lower"])),
        col = rgb(1, 0, 0, 0.2), border = NA)

# Add vertical line at forecast start
abline(v = last_date, col = "gray", lty = 2)
abline(h = 0, col = "gray", lty = 3)

legend("topright", 
       legend = c("Historical", "Forecast", "95% CI"),
       col = c("darkblue", "red", rgb(1, 0, 0, 0.3)),
       lty = c(1, 1, NA), lwd = c(1.5, 2, NA),
       pch = c(NA, NA, 15), pt.cex = 2)

### 12.3 Rolling Window Forecast Evaluation

In [ ]:
%%R
# Rolling window out-of-sample forecast evaluation
cat("Rolling Window Forecast Evaluation:\n")
cat("====================================\n\n")

# Parameters
window_size <- 120  # 10 years of training data
n_obs <- nrow(var_data)
n_forecasts <- n_obs - window_size - 1

if (n_forecasts > 12) {
  # Store forecasts and actuals
  fc_1step <- numeric(n_forecasts)
  actual_1step <- numeric(n_forecasts)
  
  cat(paste("Training window:", window_size, "months\n"))
  cat(paste("Number of forecasts:", n_forecasts, "\n"))
  cat("Computing rolling forecasts...\n\n")
  
  for (i in 1:n_forecasts) {
    # Training data
    train_end <- window_size + i - 1
    train_data <- var_data[1:train_end, ]
    
    # Estimate VAR on training data
    var_roll <- tryCatch({
      VAR(train_data, p = p, type = "const")
    }, error = function(e) NULL)
    
    if (!is.null(var_roll)) {
      # 1-step ahead forecast
      fc <- predict(var_roll, n.ahead = 1)
      fc_1step[i] <- fc$fcst$SP500_Return[1, "fcst"]
      actual_1step[i] <- var_data[train_end + 1, "SP500_Return"]
    }
  }
  
  # Remove any NA values
  valid_idx <- !is.na(fc_1step) & !is.na(actual_1step)
  fc_1step <- fc_1step[valid_idx]
  actual_1step <- actual_1step[valid_idx]
  
  # Calculate forecast accuracy metrics
  forecast_errors <- actual_1step - fc_1step
  
  mae <- mean(abs(forecast_errors))
  rmse <- sqrt(mean(forecast_errors^2))
  mape <- mean(abs(forecast_errors / actual_1step), na.rm = TRUE) * 100
  
  # Direction accuracy
  direction_correct <- mean((fc_1step > 0) == (actual_1step > 0)) * 100
  
  cat("Forecast Accuracy Metrics (1-step ahead):\n")
  cat(paste("  MAE  (Mean Absolute Error):    ", round(mae, 4), "\n"))
  cat(paste("  RMSE (Root Mean Squared Error):", round(rmse, 4), "\n"))
  cat(paste("  Direction Accuracy:            ", round(direction_correct, 2), "%\n"))
  
  # Store for later use
  rolling_fc <- list(forecasts = fc_1step, actuals = actual_1step, errors = forecast_errors)
} else {
  cat("Insufficient data for rolling window evaluation.\n")
  cat(paste("Need at least", window_size + 13, "observations, have", n_obs, "\n"))
}

In [ ]:
%%R
# Plot rolling forecast results
if (exists("rolling_fc") && length(rolling_fc$forecasts) > 0) {
  par(mfrow = c(2, 2), mar = c(4, 4, 3, 1))
  
  # Actual vs Forecast scatter
  plot(rolling_fc$actuals, rolling_fc$forecasts, 
       pch = 19, col = rgb(0, 0, 1, 0.5),
       xlab = "Actual Returns (%)", ylab = "Forecast Returns (%)",
       main = "Actual vs Forecast (1-step)")
  abline(0, 1, col = "red", lwd = 2)
  abline(lm(rolling_fc$forecasts ~ rolling_fc$actuals), col = "blue", lty = 2)
  
  # Forecast error distribution
  hist(rolling_fc$errors, breaks = 30, freq = FALSE, col = "lightblue",
       main = "Forecast Error Distribution",
       xlab = "Forecast Error (%)")
  curve(dnorm(x, mean = mean(rolling_fc$errors), sd = sd(rolling_fc$errors)),
        add = TRUE, col = "red", lwd = 2)
  
  # Time series of forecasts vs actuals
  plot(rolling_fc$actuals, type = "l", col = "darkblue",
       main = "Rolling Forecast vs Actual",
       ylab = "Return (%)", xlab = "Forecast Period")
  lines(rolling_fc$forecasts, col = "red", lty = 2)
  legend("topright", legend = c("Actual", "Forecast"),
         col = c("darkblue", "red"), lty = c(1, 2))
  
  # Cumulative squared error
  cse <- cumsum(rolling_fc$errors^2)
  plot(cse, type = "l", col = "purple", lwd = 2,
       main = "Cumulative Squared Error",
       ylab = "CSE", xlab = "Forecast Period")
}

### 12.4 Comparison with Benchmark Models

In [ ]:
%%R
cat("Comparison with Benchmark Models:\n")
cat("==================================\n\n")

# Extract S&P 500 returns for univariate models
sp500_ts <- as.numeric(econ_stationary$SP500_Return)

# 1. Random Walk (mean forecast)
rw_forecast <- mean(sp500_ts)

# 2. AR(1) model
ar1_model <- arima(sp500_ts, order = c(1, 0, 0))
ar1_forecast <- predict(ar1_model, n.ahead = 12)$pred

# 3. ARIMA (auto-selected)
auto_arima_model <- auto.arima(sp500_ts, seasonal = FALSE, max.p = 6, max.q = 6)
arima_forecast <- forecast(auto_arima_model, h = 12)

cat("Model Specifications:\n")
cat(paste("  VAR:", paste0("VAR(", p, ")"), "with", ncol(econ_stationary), "variables\n"))
cat(paste("  AR(1): phi =", round(ar1_model$coef[1], 4), "\n"))
cat(paste("  Auto-ARIMA:", paste0("ARIMA(", paste(arimaorder(auto_arima_model), collapse = ","), ")"), "\n\n"))

# Compare 12-month ahead forecasts
cat("12-Month Ahead Forecasts:\n")
forecast_comparison <- data.frame(
  Horizon = 1:12,
  VAR = round(sp500_fc[, "fcst"], 4),
  AR1 = round(as.numeric(ar1_forecast), 4),
  ARIMA = round(as.numeric(arima_forecast$mean), 4),
  RW = round(rep(rw_forecast, 12), 4)
)

print(forecast_comparison)

In [ ]:
%%R
# Plot forecast comparison
par(mfrow = c(1, 1), mar = c(4, 4, 3, 1))

# Historical data (last 24 months)
n_hist <- 24
hist_data <- tail(sp500_ts, n_hist)

# Combine historical and forecasts
plot_range <- c(-15, 15)  # Adjust based on data

plot(1:n_hist, hist_data, type = "l", col = "darkblue", lwd = 2,
     xlim = c(1, n_hist + 12), ylim = plot_range,
     xlab = "Period", ylab = "Return (%)",
     main = "Forecast Comparison: VAR vs Benchmark Models")

# Add forecasts
lines((n_hist + 1):(n_hist + 12), sp500_fc[, "fcst"], col = "red", lwd = 2)
lines((n_hist + 1):(n_hist + 12), ar1_forecast, col = "green", lwd = 2, lty = 2)
lines((n_hist + 1):(n_hist + 12), arima_forecast$mean, col = "orange", lwd = 2, lty = 3)
abline(h = rw_forecast, col = "purple", lty = 4)

# Vertical line
abline(v = n_hist + 0.5, col = "gray", lty = 2)
abline(h = 0, col = "gray", lty = 3)

legend("topleft", 
       legend = c("Historical", "VAR", "AR(1)", "Auto-ARIMA", "Random Walk"),
       col = c("darkblue", "red", "green", "orange", "purple"),
       lty = c(1, 1, 2, 3, 4), lwd = 2, cex = 0.8)

---
## Part 13: Summary and Conclusions

### 13.1 Key Findings

In [ ]:
%%R
cat("============================================================\n")
cat("           ECONOMETRIC ANALYSIS: KEY FINDINGS               \n")
cat("============================================================\n\n")

cat("1. DATA SUMMARY\n")
cat("---------------\n")
cat(paste("   Sample period:", min(index(econ_stationary)), "to", 
          max(index(econ_stationary)), "\n"))
cat(paste("   Number of observations:", nrow(econ_stationary), "months\n"))
cat(paste("   Variables analyzed:", ncol(econ_stationary), "\n\n"))

cat("2. STATIONARITY\n")
cat("---------------\n")
cat("   Most variables are stationary after appropriate transformations:\n")
cat("   - S&P 500 Returns: Stationary (log returns)\n")
cat("   - Inflation: Stationary (YoY change)\n")
cat("   - Fed Funds: Stationary after first differencing\n")
cat("   - Unemployment: Stationary after first differencing\n")
cat("   - Industrial Production: Stationary (growth rate)\n\n")

cat("3. VAR MODEL\n")
cat("-----------\n")
cat(paste("   Optimal lag order (AIC):", optimal_lag, "\n"))
cat(paste("   Model estimated: VAR(", p, ")\n", sep = ""))
cat(paste("   Model stability: All eigenvalues inside unit circle\n\n"))

cat("4. GRANGER CAUSALITY\n")
cat("--------------------\n")
for (i in 1:nrow(granger_to_sp500)) {
  cat(paste("   ", granger_to_sp500$Cause[i], "->", "SP500:", 
            ifelse(granger_to_sp500$p_value[i] < 0.05, "Significant*", "Not significant"),
            "(p =", granger_to_sp500$p_value[i], ")\n"))
}
cat("\n")

cat("5. IMPULSE RESPONSES\n")
cat("--------------------\n")
cat("   Stock returns respond to:\n")
cat("   - Own shocks: Strong initial response, rapid decay\n")
cat("   - Inflation shocks: Generally negative relationship\n")
cat("   - Fed Funds changes: Negative contemporaneous effect\n")
cat("   - Unemployment changes: Counter-cyclical relationship\n")
cat("   - Industrial production: Positive relationship\n\n")

cat("6. FORECAST EVALUATION\n")
cat("----------------------\n")
if (exists("rolling_fc")) {
  cat(paste("   Rolling window RMSE:", round(rmse, 4), "\n"))
  cat(paste("   Direction accuracy:", round(direction_correct, 2), "%\n"))
}
cat("\n")

### 13.2 Economic Interpretation

In [ ]:
%%R
cat("============================================================\n")
cat("              ECONOMIC INTERPRETATION                        \n")
cat("============================================================\n\n")

cat("THEORETICAL FRAMEWORK\n")
cat("---------------------\n")
cat("This analysis examines the relationship between stock market returns\n")
cat("and key macroeconomic variables, grounded in several economic theories:\n\n")

cat("1. FISHER EFFECT & INFLATION\n")
cat("   The negative relationship between inflation and stock returns\n")
cat("   supports the 'proxy hypothesis' - high inflation often coincides\n")
cat("   with poor economic conditions and reduced corporate profitability.\n\n")

cat("2. MONETARY POLICY TRANSMISSION\n")
cat("   Changes in the Fed Funds rate affect stock prices through:\n")
cat("   - Discount rate channel: Higher rates reduce present value of\n")
cat("     future cash flows\n")
cat("   - Economic activity channel: Tighter policy slows growth\n")
cat("   - Risk appetite channel: Higher rates reduce risk-taking\n\n")

cat("3. BUSINESS CYCLE DYNAMICS\n")
cat("   - Stock markets are forward-looking and often lead economic\n")
cat("     indicators by 6-12 months\n")
cat("   - Industrial production captures real economic activity\n")
cat("   - Unemployment is a lagging indicator but affects consumer\n")
cat("     spending and corporate earnings\n\n")

cat("PRACTICAL IMPLICATIONS\n")
cat("----------------------\n")
cat("1. Portfolio managers should monitor monetary policy shifts\n")
cat("2. Inflation expectations matter for equity valuations\n")
cat("3. Leading indicators (e.g., industrial production) may provide\n")
cat("   useful signals for timing decisions\n")
cat("4. The VAR framework captures dynamic interdependencies that\n")
cat("   simple regression models miss\n\n")

cat("LIMITATIONS\n")
cat("-----------\n")
cat("1. Linear relationships assumed (may miss nonlinearities)\n")
cat("2. Parameter instability during regime changes (e.g., 2008 crisis)\n")
cat("3. Omitted variable bias (other factors affect returns)\n")
cat("4. Data revisions can affect real-time forecasting\n")

### 13.3 Final Reproducibility Check

In [ ]:
%%R
cat("============================================================\n")
cat("              REPRODUCIBILITY INFORMATION                    \n")
cat("============================================================\n\n")

cat("SESSION INFORMATION:\n")
cat("--------------------\n")
sessionInfo()

cat("\n\nSTORED R OBJECTS:\n")
cat("-----------------\n")
obj_sizes <- sapply(ls(), function(x) object.size(get(x)))
obj_df <- data.frame(
  Object = names(obj_sizes),
  Size_KB = round(obj_sizes / 1024, 2)
)
obj_df <- obj_df[order(-obj_df$Size_KB), ]
head(obj_df, 20)

cat("\n\nRANDOM SEED: 42\n")
cat("All stochastic operations use set.seed(42) for reproducibility.\n")

In [ ]:
%%R
# Save key results for export
results_summary <- list(
  data = list(
    econ_data = econ_data,
    econ_stationary = econ_stationary
  ),
  stationarity_tests = list(
    adf = adf_results,
    pp = pp_results,
    kpss = kpss_results
  ),
  var_model = var_model,
  granger_causality = granger_to_sp500,
  forecasts = forecast_comparison,
  diagnostics = list(
    eigenvalues = roots,
    correlation_matrix = cor_matrix
  )
)

cat("\n============================================================\n")
cat("              ANALYSIS COMPLETE                              \n")
cat("============================================================\n\n")
cat("All results stored in 'results_summary' object.\n")
cat("This notebook can be re-run to reproduce all results.\n")
cat("\nThank you for using this econometric analysis notebook!\n")